# Maricopa County Clinic Need — ZCTA Choropleth

This notebook converts the script-style `need_map.py` into a Jupyter-friendly workflow. It produces:
- A static PNG choropleth (`maricopa_choropleth_census_v2.png` by default)
- An interactive HTML map (`maricopa_choropleth_census_v2.html` by default)

Usage notes: edit the `csv_path`, `png_path`, and `html_path` variables in the *Parameters* cell and then run the cells top-to-bottom.


In [5]:
# Imports
import os
from pathlib import Path
import requests
import geopandas as gpd
import pandas as pd


In [6]:
# Constants and helper functions adapted for notebook use
CB_ZCTA_URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_zcta520_500k.zip"

def get_cb_zcta_zip_path(base_dir: Path) -> Path:
    "Return path where the ZCTA zip should be stored relative to a base directory."
    return Path(base_dir) / "data" / "cb_2020_us_zcta520_500k.zip"

def download_if_missing(url: str, out_path: Path) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and out_path.stat().st_size > 0:
        print("Using existing file: {out_path}")
        return out_path

    print(f"Downloading ZCTA shapefile → {out_path} ...")
    try:
        r = requests.get(url, timeout=180)
        r.raise_for_status()
        out_path.write_bytes(r.content)
        print(f"Downloaded successfully: {out_path}")
        return out_path
    except requests.exceptions.RequestException as e:
        print(f"Failed to download from {url}: {e}")
        raise Exception("Could not download shapefile. Please manually download from the Census cartographic boundary page and place it in the `data/` directory.")

def load_zcta_shapes(zip_path: Path, keep_zctas: pd.Series) -> gpd.GeoDataFrame:
    "Read the zipped ZCTA shapefile and filter to the provided ZCTAs."
    gdf = gpd.read_file(f"zip://{zip_path}")
    keep = set(keep_zctas.astype(str).str.zfill(5))
    gdf = gdf[gdf["ZCTA5CE20"].isin(keep)].copy()
    print(f"Filtered to {len(gdf)} ZCTAs for Maricopa County")
    return gdf


In [7]:
# Plotting helper functions (static PNG + interactive HTML)
def make_static_png(gdf: gpd.GeoDataFrame, out_png: str):
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    gdf.to_crs(3857).plot(
        column="need_score",
        scheme="Quantiles",
        k=5,
        legend=True,
        cmap="OrRd",
        edgecolor="white",
        linewidth=0.3,
        missing_kwds={"color": "lightgrey", "label": "No data"},
        ax=ax,
        legend_kwds={'title': 'Clinic Need Score\\n(Income + Population)', 'title_fontsize': 12, 'fontsize': 10},
    )
    ax.set_axis_off()
    title = f"Healthcare Clinic Need Analysis - Maricopa County, AZ\\nBased on Income Level + Population Density + Chronic Disease Prevalence"
    ax.set_title(title, pad=25, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved static PNG → {out_png}")

def make_interactive_html(gdf: gpd.GeoDataFrame, out_html: str):
    gdf_wgs84 = gdf.to_crs(4326)
    gdf_wgs84["median_income_fmt"] = gdf_wgs84["census_B19013_001E"].map(lambda x: f"${x:,.0f}" if pd.notnull(x) else "N/A")
    gdf_wgs84["population_fmt"] = gdf_wgs84["census_B01001_001E"].map(lambda x: f"{int(x):,}" if pd.notnull(x) else "N/A")
    gdf_wgs84["need_score_fmt"] = gdf_wgs84["need_score"].map(lambda x: f"{x:.3f}" if pd.notnull(x) else "N/A")

    m = gdf_wgs84.explore(
        column="need_score",
        scheme="Quantiles",
        k=5,
        tooltip=["ZCTA5CE20", "population_fmt", "need_score_fmt"],
        popup=["ZCTA5CE20", "population_fmt", "need_score_fmt"],
        name="Clinic Need",
        legend=True,
        cmap="OrRd",
        style_kwds={"fillOpacity": 0.7, "weight": 1, "color": "white"},
    )
    m.save(out_html)
    print(f"Saved interactive HTML → {out_html}")

In [8]:
# Parameters (edit these in the notebook)
base_dir = Path('.').resolve()
csv_path = base_dir / 'data/maricopa_healthcare_standardized_needscore.csv'
png_path = base_dir / 'map.png'
html_path = base_dir / 'docs/map.html'

# Load CSV
print(f"Loading data from {csv_path} ...")
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")
df = pd.read_csv(csv_path, dtype={"zcta": str})
if 'need_score' not in df.columns:
    raise RuntimeError("CSV is missing 'need_score'. Make sure the preprocessing step was run.")
df['zcta'] = df['zcta'].astype(str).str.zfill(5)
print(f"Loaded {len(df)} rows from CSV")

# Download / prepare ZCTA shapes
try:
    zip_path = download_if_missing(CB_ZCTA_URL, get_cb_zcta_zip_path(base_dir))
    zcta_gdf = load_zcta_shapes(zip_path, df['zcta'])
except Exception as e:
    print(f"Error loading shapefiles: {e}")
    print("If automatic download fails, manually download the 2020 ZCTA cartographic boundary zip and place it in the data/ folder.")
    raise

# Merge attributes and create outputs
gdf = zcta_gdf.merge(df, left_on='ZCTA5CE20', right_on='zcta', how='left', validate='one_to_one')
gdf = gdf.sort_values('need_score', ascending=False)
print(f"Final dataset has {len(gdf)} ZCTAs with geographic boundaries")

# Create files (wrap in try/except to allow inspection on failure)
try:
    make_static_png(gdf, str(png_path))
except Exception as e:
    print(f"Error creating PNG: {e}")

try:
    make_interactive_html(gdf, str(html_path))
except Exception as e:
    print(f"Error creating HTML: {e}")

Loading data from /Users/ainsleyree/Desktop/AZEquiScope/data/maricopa_healthcare_standardized_needscore.csv ...
Loaded 128 rows from CSV
Using existing file: {out_path}
Filtered to 128 ZCTAs for Maricopa County
Final dataset has 128 ZCTAs with geographic boundaries
Filtered to 128 ZCTAs for Maricopa County
Final dataset has 128 ZCTAs with geographic boundaries
Saved static PNG → /Users/ainsleyree/Desktop/AZEquiScope/map.png
Saved interactive HTML → /Users/ainsleyree/Desktop/AZEquiScope/docs/map.html
Saved static PNG → /Users/ainsleyree/Desktop/AZEquiScope/map.png
Saved interactive HTML → /Users/ainsleyree/Desktop/AZEquiScope/docs/map.html
